# 02 · Curvas y valorización por flujos

Orden (Anexo 2, regla 4): bono fijo simple → flotante con una forward → portafolio → **test de calibración** (valor teórico = monto de mercado al 31-12-2025).

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from optimum import paths
from optimum.cleaning import load_processed
from optimum.reporting.figures import COLORS, new_figure, save_fig
from optimum.reporting.tables import save_table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

In [ ]:
from optimum.curves import ZeroCurve, implied_forward

hist = load_processed("market_history")
base = hist.loc["2025-12-31"]
curves = {c: ZeroCurve.from_market_row(base, c) for c in ["PEN", "USD"]}
t = np.linspace(0, 20, 201)
fig, ax = new_figure()
for c, cv in curves.items():
    ax.plot(t, 100 * cv.zero(t), color=COLORS[c], label=c)
    ax.scatter(cv.tenors, 100 * cv.rates, color=COLORS[c], s=12)
ax.set(xlabel="Años", ylabel="Tasa cero (%)", title="Curvas spot al 31-12-2025")
ax.legend()
save_fig(fig, "curvas_spot_base")

In [ ]:
# Chequeo: forwards implícitas vs. forwards del Excel
from optimum.curves import NODE_YEARS, SEGMENT_BOUNDS

rows = []
for c, cv in curves.items():
    for s, (a, b) in SEGMENT_BOUNDS.items():
        t1, t2 = max(NODE_YEARS[a], 1e-9), NODE_YEARS[b]
        rows.append(
            {"ccy": c, "seg": s, "implícita": implied_forward(cv, t1, t2), "excel": base[f"{c}_FWD_{s}"]}
        )
pd.DataFrame(rows).assign(dif_pb=lambda d: 1e4 * (d["implícita"] - d["excel"])).round(5)

## TODO: valorizador (`src/optimum/valuation.py`)

1. `price_fixed` con A02 → comparar con 180.
2. `price_floating` con A03.
3. Todos los instrumentos → tabla valor teórico vs. mercado.
4. Mover a tests lo que ya funcione.